# 03 - Dispersão de Pontuação por Temporada

Análise do desvio padrão de pontos finais: quais temporadas foram mais equilibradas?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df.dropna(subset=["rodata"])
df["rodata"] = df["rodata"].astype(int)
df = df[df["ano"] >= 2003].copy()

In [2]:
def classificacao_final(df_season):
    """Retorna a pontuação final de cada time em uma temporada."""
    teams = set(df_season["mandante"].unique()) | set(df_season["visitante"].unique())
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    
    for _, jogo in df_season.iterrows():
        m, v = jogo["mandante"], jogo["visitante"]
        gm, gv = jogo["mandante_Placar"], jogo["visitante_Placar"]
        if pd.isna(gm) or pd.isna(gv):
            continue
        gm, gv = int(gm), int(gv)
        saldo[m] += gm - gv
        saldo[v] += gv - gm
        if gm > gv:
            pontos[m] += 3
            vitorias[m] += 1
        elif gm < gv:
            pontos[v] += 3
            vitorias[v] += 1
        else:
            pontos[m] += 1
            pontos[v] += 1
    
    ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
    return [(t, pontos[t], i+1) for i, t in enumerate(ranking)]

In [3]:
# Calcular classificação final de cada temporada
todas_classificacoes = []

for ano in sorted(df["ano"].unique()):
    df_season = df[df["ano"] == ano]
    if df_season["rodata"].max() < 30:
        continue
    classif = classificacao_final(df_season)
    for time, pts, pos in classif:
        todas_classificacoes.append({"ano": ano, "time": time, "pontos": pts, "posicao": pos})

df_class = pd.DataFrame(todas_classificacoes)
df_class.head()

,ano,time,pontos,posicao
0,2003,Cruzeiro,100,1
1,2003,Santos,87,2
2,2003,Sao Paulo,78,3
3,2003,Coritiba,73,4
4,2003,Atletico-MG,72,5


In [4]:
# Desvio padrão e amplitude por temporada
stats = df_class.groupby("ano")["pontos"].agg(["std", "mean", "min", "max"]).reset_index()
stats["amplitude"] = stats["max"] - stats["min"]

# Gap entre 1º e 2º lugar
gaps = []
for ano in df_class["ano"].unique():
    season = df_class[df_class["ano"] == ano].sort_values("posicao")
    pts_1 = season.iloc[0]["pontos"]
    pts_2 = season.iloc[1]["pontos"]
    pts_4 = season.iloc[3]["pontos"] if len(season) > 3 else None
    pts_17 = season.iloc[16]["pontos"] if len(season) > 16 else None
    gaps.append({"ano": ano, "gap_1_2": pts_1 - pts_2, "pts_campeao": pts_1,
                 "pts_4": pts_4, "pts_17": pts_17})

df_gaps = pd.DataFrame(gaps)
stats = stats.merge(df_gaps, on="ano")

In [5]:
# Gráfico 1: Desvio padrão por temporada
fig = make_subplots(rows=2, cols=1, subplot_titles=[
    "Desvio Padrão da Pontuação Final",
    "Gap entre 1º e 2º Lugar (pontos)"
], vertical_spacing=0.15)

fig.add_trace(go.Bar(
    x=stats["ano"], y=stats["std"],
    marker_color="#2ecc71",
    name="Desvio Padrão",
    hovertemplate="%{x}: σ = %{y:.1f}<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=stats["ano"], y=stats["gap_1_2"],
    marker_color="#e74c3c",
    name="Gap 1º-2º",
    hovertemplate="%{x}: %{y} pts de diferença<extra></extra>"
), row=2, col=1)

fig.update_layout(
    title="Equilíbrio do Brasileirão por Temporada<br><sup>Série A (2003-presente)</sup>",
    template="plotly_white",
    height=700,
    width=900,
    showlegend=False
)
fig.update_yaxes(title_text="Desvio Padrão", row=1, col=1)
fig.update_yaxes(title_text="Pontos", row=2, col=1)

fig.show()
_path = "../charts/desvio_padrao.html"
fig.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Métricas de dispersão e competitividade por temporada. O desvio padrão mede a heterogeneidade da pontuação entre os clubes, enquanto o gap entre 1º e 2º lugar quantifica a dominância do campeão. Valores menores indicam temporadas mais equilibradas."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/desvio_padrao.html")

Gráfico exportado: charts/desvio_padrao.html


In [6]:
# Gráfico 2: Box plot de pontos por temporada
fig2 = px.box(df_class, x="ano", y="pontos",
              title="Distribuição de Pontos por Temporada<br><sup>Box plot - cada ponto é um time</sup>",
              labels={"ano": "Ano", "pontos": "Pontos"},
              color_discrete_sequence=["#27ae60"])
fig2.update_layout(template="plotly_white", width=900, height=500)

fig2.show()
_path = "../charts/distribuicao_pontos.html"
fig2.write_html(_path, include_plotlyjs="cdn")

# Add description
_desc = "Box plots exibindo a distribuição de pontos por temporada com mediana, quartis e outliers. A amplitude interquartil e a presença de valores extremos permitem avaliar a assimetria e o grau de concentração de pontuação."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print("Gráfico exportado: charts/distribuicao_pontos.html")

Gráfico exportado: charts/distribuicao_pontos.html


In [7]:
# Insights
mais_equilibrada = stats.loc[stats["std"].idxmin()]
menos_equilibrada = stats.loc[stats["std"].idxmax()]
maior_gap = stats.loc[stats["gap_1_2"].idxmax()]

print(f"Temporada mais equilibrada: {int(mais_equilibrada['ano'])} (σ = {mais_equilibrada['std']:.1f})")
print(f"Temporada menos equilibrada: {int(menos_equilibrada['ano'])} (σ = {menos_equilibrada['std']:.1f})")
print(f"Maior gap 1º-2º: {int(maior_gap['ano'])} ({int(maior_gap['gap_1_2'])} pontos)")

Temporada mais equilibrada: 2017 (σ = 9.1)
Temporada menos equilibrada: 2021 (σ = 26.7)
Maior gap 1º-2º: 2019 (16 pontos)
